In [ ]:
import kaggle_benchmarks as kbench
from IPython.display import HTML, display

# ================================================================================
# ASYMMETRIC MAZE COOPERATION BENCHMARK
# Two instances of the same LLM cooperate under asymmetric information.
# Player 1 (Navigator): inside the maze, sees only adjacent walls.
# Player 2 (Cartographer): has the full map, doesn't know Player 1's position.
# ================================================================================

# --------------------------------------------------------------------------------
# 1. THE MAZE SUITE — Fixed, Hardcoded, Identical for All Models
# --------------------------------------------------------------------------------
# Wall encoding: 1 = solid wall, 0 = open passage
# Coordinates: (row, col), 0-indexed from top-left
# All mazes are perfect mazes (exactly one path between any two cells)
# --------------------------------------------------------------------------------

MAZE_SUITE = [
    {
        "name": "Maze 1: Warmup",
        "size": (3, 3),
        "start": (2, 0),
        "exit": (0, 2),
        "optimal": 4,
        "limit": 15,
        "ascii": (
            "+--+--+--+\n"
            "|     | E|\n"
            "+  +--+  +\n"
            "|     |  |\n"
            "+--+  +  +\n"
            "|S |     |\n"
            "+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 0, "W": 1},
            (0, 1): {"N": 1, "S": 1, "E": 0, "W": 0},
            (0, 2): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 0, "S": 1, "E": 0, "W": 1},
            (1, 1): {"N": 1, "S": 0, "E": 0, "W": 0},
            (1, 2): {"N": 0, "S": 1, "E": 1, "W": 0},
            (2, 0): {"N": 1, "S": 1, "E": 0, "W": 1},
            (2, 1): {"N": 0, "S": 1, "E": 0, "W": 0},
            (2, 2): {"N": 1, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 2: Easy",
        "size": (4, 4),
        "start": (3, 0),
        "exit": (0, 3),
        "optimal": 8,
        "limit": 20,
        "ascii": (
            "+--+--+--+--+\n"
            "|         E |\n"
            "+--+--+--+  +\n"
            "|     |     |\n"
            "+--+  +  +--+\n"
            "|     |     |\n"
            "+  +--+--+  +\n"
            "|S          |\n"
            "+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 1, "E": 0, "W": 1}, (0, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 3): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 1, "S": 1, "E": 0, "W": 1}, (1, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (1, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (1, 3): {"N": 0, "S": 1, "E": 1, "W": 0},
            (2, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (2, 2): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 3): {"N": 1, "S": 0, "E": 1, "W": 0},
            (3, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (3, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (3, 3): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 3: Localization",
        "size": (5, 5),
        "start": (0, 0),
        "exit": (4, 4),
        "optimal": 12,
        "limit": 25,
        "ascii": (
            "+--+--+--+--+--+\n"
            "|S |        |  |\n"
            "+  +  +--+  +  +\n"
            "|     |  |     |\n"
            "+--+--+  +--+  +\n"
            "|        |     |\n"
            "+  +--+  +  +--+\n"
            "|  |  |  |     |\n"
            "+  +  +  +--+  +\n"
            "|     |      E |\n"
            "+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 3): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 4): {"N": 1, "S": 0, "E": 1, "W": 1},
            (1, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 2): {"N": 1, "S": 0, "E": 1, "W": 1}, (1, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 4): {"N": 0, "S": 0, "E": 1, "W": 0},
            (2, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 2): {"N": 0, "S": 0, "E": 1, "W": 0}, (2, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 4): {"N": 0, "S": 1, "E": 1, "W": 0},
            (3, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 1): {"N": 1, "S": 0, "E": 1, "W": 1}, (3, 2): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 4): {"N": 1, "S": 0, "E": 1, "W": 0},
            (4, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (4, 2): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 4): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 4: More Ambiguity",
        "size": (5, 5),
        "start": (2, 0),
        "exit": (2, 4),
        "optimal": 10,
        "limit": 30,
        "ascii": (
            "+--+--+--+--+--+\n"
            "|  |           |\n"
            "+  +  +--+--+  +\n"
            "|  |     |  |  |\n"
            "+  +--+  +  +  +\n"
            "|S       |  |E |\n"
            "+--+--+--+  +  +\n"
            "|     |        |\n"
            "+  +--+  +--+--+\n"
            "|              |\n"
            "+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 4): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 2): {"N": 1, "S": 0, "E": 1, "W": 0}, (1, 3): {"N": 1, "S": 0, "E": 1, "W": 1}, (1, 4): {"N": 0, "S": 0, "E": 1, "W": 1},
            (2, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (2, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 4): {"N": 0, "S": 0, "E": 1, "W": 1},
            (3, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 1): {"N": 1, "S": 1, "E": 1, "W": 0}, (3, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 3): {"N": 0, "S": 1, "E": 0, "W": 0}, (3, 4): {"N": 0, "S": 1, "E": 1, "W": 0},
            (4, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 2): {"N": 0, "S": 1, "E": 0, "W": 0}, (4, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 4): {"N": 1, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 5: Medium",
        "size": (6, 6),
        "start": (5, 3),
        "exit": (0, 3),
        "optimal": 15,
        "limit": 35,
        "ascii": (
            "+--+--+--+--+--+--+\n"
            "|     |   E |     |\n"
            "+--+  +--+  +  +  +\n"
            "|  |     |     |  |\n"
            "+  +--+  +--+--+  +\n"
            "|     |           |\n"
            "+--+  +--+--+--+  +\n"
            "|     |  |     |  |\n"
            "+  +  +  +  +  +  +\n"
            "|  |     |  |  |  |\n"
            "+  +--+--+  +  +  +\n"
            "|         S |     |\n"
            "+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 1, "E": 0, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 2): {"N": 1, "S": 1, "E": 0, "W": 1}, (0, 3): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 5): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (1, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 2): {"N": 1, "S": 0, "E": 1, "W": 0}, (1, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 5): {"N": 0, "S": 0, "E": 1, "W": 1},
            (2, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (2, 2): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 5): {"N": 0, "S": 0, "E": 1, "W": 0},
            (3, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 1): {"N": 0, "S": 0, "E": 1, "W": 0}, (3, 2): {"N": 1, "S": 0, "E": 1, "W": 1}, (3, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 4): {"N": 1, "S": 0, "E": 1, "W": 0}, (3, 5): {"N": 0, "S": 0, "E": 1, "W": 1},
            (4, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (4, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 4): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 5): {"N": 0, "S": 0, "E": 1, "W": 1},
            (5, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (5, 4): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 5): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 6: Far Apart",
        "size": (6, 6),
        "start": (0, 5),
        "exit": (5, 0),
        "optimal": 14,
        "limit": 35,
        "ascii": (
            "+--+--+--+--+--+--+\n"
            "|  |           |S |\n"
            "+  +--+  +--+  +  +\n"
            "|        |     |  |\n"
            "+--+--+--+  +--+  +\n"
            "|        |  |     |\n"
            "+--+  +  +  +  +--+\n"
            "|     |  |  |     |\n"
            "+  +--+--+  +  +  +\n"
            "|     |     |  |  |\n"
            "+  +  +  +--+--+  +\n"
            "|E |              |\n"
            "+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 1, "E": 0, "W": 1}, (0, 2): {"N": 1, "S": 0, "E": 0, "W": 0}, (0, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 4): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 5): {"N": 1, "S": 0, "E": 1, "W": 1},
            (1, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (1, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 5): {"N": 0, "S": 0, "E": 1, "W": 1},
            (2, 0): {"N": 1, "S": 1, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 0, "E": 0, "W": 0}, (2, 2): {"N": 1, "S": 0, "E": 1, "W": 0}, (2, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 5): {"N": 0, "S": 1, "E": 1, "W": 0},
            (3, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (3, 2): {"N": 0, "S": 1, "E": 1, "W": 1}, (3, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 4): {"N": 0, "S": 0, "E": 0, "W": 1}, (3, 5): {"N": 1, "S": 0, "E": 1, "W": 0},
            (4, 0): {"N": 0, "S": 0, "E": 0, "W": 1}, (4, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (4, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (4, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (4, 4): {"N": 0, "S": 1, "E": 1, "W": 1}, (4, 5): {"N": 0, "S": 0, "E": 1, "W": 1},
            (5, 0): {"N": 0, "S": 1, "E": 1, "W": 1}, (5, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 2): {"N": 0, "S": 1, "E": 0, "W": 0}, (5, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 5): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 7: Hard Planning",
        "size": (7, 7),
        "start": (6, 0),
        "exit": (0, 6),
        "optimal": 18,
        "limit": 45,
        "ascii": (
            "+--+--+--+--+--+--+--+\n"
            "|  |  |            E |\n"
            "+  +  +  +--+--+--+  +\n"
            "|  |  |  |           |\n"
            "+  +  +  +  +--+--+  +\n"
            "|  |  |  |     |     |\n"
            "+  +  +  +--+  +--+--+\n"
            "|  |  |     |        |\n"
            "+  +  +--+  +--+--+  +\n"
            "|  |     |        |  |\n"
            "+  +  +--+  +--+--+  +\n"
            "|  |        |        |\n"
            "+  +--+--+--+  +--+  +\n"
            "|S             |     |\n"
            "+--+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 6): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 1): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 2): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (1, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 6): {"N": 0, "S": 0, "E": 1, "W": 0},
            (2, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 1): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 2): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 4): {"N": 1, "S": 0, "E": 1, "W": 0}, (2, 5): {"N": 1, "S": 1, "E": 0, "W": 1}, (2, 6): {"N": 0, "S": 1, "E": 1, "W": 0},
            (3, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 1): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 2): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 3): {"N": 1, "S": 0, "E": 1, "W": 0}, (3, 4): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (3, 6): {"N": 1, "S": 0, "E": 1, "W": 0},
            (4, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 1): {"N": 0, "S": 0, "E": 0, "W": 1}, (4, 2): {"N": 1, "S": 1, "E": 1, "W": 0}, (4, 3): {"N": 0, "S": 0, "E": 0, "W": 1}, (4, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 5): {"N": 1, "S": 1, "E": 1, "W": 0}, (4, 6): {"N": 0, "S": 0, "E": 1, "W": 1},
            (5, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (5, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (5, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (5, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 6): {"N": 0, "S": 0, "E": 1, "W": 0},
            (6, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (6, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (6, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (6, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (6, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (6, 5): {"N": 1, "S": 1, "E": 0, "W": 1}, (6, 6): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 8: High Ambiguity",
        "size": (7, 7),
        "start": (3, 0),
        "exit": (0, 6),
        "optimal": 21,
        "limit": 45,
        "ascii": (
            "+--+--+--+--+--+--+--+\n"
            "|  |  |            E |\n"
            "+  +  +  +--+--+--+  +\n"
            "|  |     |           |\n"
            "+  +--+--+  +--+--+  +\n"
            "|     |     |  |     |\n"
            "+--+  +  +--+  +  +--+\n"
            "|S |     |     |  |  |\n"
            "+  +--+--+  +--+  +  +\n"
            "|  |        |     |  |\n"
            "+  +  +--+  +  +--+  +\n"
            "|  |  |  |  |     |  |\n"
            "+  +  +  +  +--+  +  +\n"
            "|        |           |\n"
            "+--+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 6): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (1, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 6): {"N": 0, "S": 0, "E": 1, "W": 0},
            (2, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (2, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (2, 4): {"N": 1, "S": 0, "E": 1, "W": 1}, (2, 5): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 6): {"N": 0, "S": 1, "E": 1, "W": 0},
            (3, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (3, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (3, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (3, 5): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 6): {"N": 1, "S": 0, "E": 1, "W": 1},
            (4, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 1): {"N": 1, "S": 0, "E": 0, "W": 1}, (4, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 3): {"N": 0, "S": 0, "E": 1, "W": 0}, (4, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (4, 5): {"N": 0, "S": 1, "E": 1, "W": 0}, (4, 6): {"N": 0, "S": 0, "E": 1, "W": 1},
            (5, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (5, 1): {"N": 0, "S": 0, "E": 1, "W": 1}, (5, 2): {"N": 1, "S": 0, "E": 1, "W": 1}, (5, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (5, 4): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 5): {"N": 1, "S": 0, "E": 1, "W": 0}, (5, 6): {"N": 0, "S": 0, "E": 1, "W": 1},
            (6, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (6, 1): {"N": 0, "S": 1, "E": 0, "W": 0}, (6, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (6, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (6, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (6, 5): {"N": 0, "S": 1, "E": 0, "W": 0}, (6, 6): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 9: Very Hard",
        "size": (8, 8),
        "start": (7, 4),
        "exit": (0, 0),
        "optimal": 25,
        "limit": 55,
        "ascii": (
            "+--+--+--+--+--+--+--+--+\n"
            "|E |  |                 |\n"
            "+  +  +  +--+--+  +--+--+\n"
            "|  |     |     |        |\n"
            "+  +  +--+--+  +--+--+  +\n"
            "|  |              |     |\n"
            "+  +--+  +--+--+--+  +--+\n"
            "|  |     |        |     |\n"
            "+  +--+  +  +--+  +--+  +\n"
            "|     |  |  |  |        |\n"
            "+--+  +--+  +  +--+--+  +\n"
            "|  |        |           |\n"
            "+  +--+--+--+  +--+--+--+\n"
            "|     |        |     |  |\n"
            "+  +  +  +--+--+  +  +  +\n"
            "|  |         S    |     |\n"
            "+--+--+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 1, "W": 1}, (0, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 5): {"N": 1, "S": 0, "E": 0, "W": 0}, (0, 6): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 7): {"N": 1, "S": 1, "E": 1, "W": 0},
            (1, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 1): {"N": 0, "S": 0, "E": 0, "W": 1}, (1, 2): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 3): {"N": 1, "S": 1, "E": 0, "W": 1}, (1, 4): {"N": 1, "S": 0, "E": 1, "W": 0}, (1, 5): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 6): {"N": 1, "S": 1, "E": 0, "W": 0}, (1, 7): {"N": 1, "S": 0, "E": 1, "W": 0},
            (2, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (2, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 2): {"N": 1, "S": 0, "E": 0, "W": 0}, (2, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 4): {"N": 0, "S": 1, "E": 0, "W": 0}, (2, 5): {"N": 1, "S": 1, "E": 1, "W": 0}, (2, 6): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 7): {"N": 0, "S": 1, "E": 1, "W": 0},
            (3, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 1): {"N": 1, "S": 1, "E": 0, "W": 1}, (3, 2): {"N": 0, "S": 0, "E": 1, "W": 0}, (3, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (3, 5): {"N": 1, "S": 0, "E": 1, "W": 0}, (3, 6): {"N": 0, "S": 1, "E": 0, "W": 1}, (3, 7): {"N": 1, "S": 0, "E": 1, "W": 0},
            (4, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (4, 2): {"N": 0, "S": 1, "E": 1, "W": 1}, (4, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 4): {"N": 1, "S": 0, "E": 1, "W": 1}, (4, 5): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 6): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 7): {"N": 0, "S": 0, "E": 1, "W": 0},
            (5, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (5, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (5, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (5, 4): {"N": 0, "S": 0, "E": 0, "W": 1}, (5, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 6): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 7): {"N": 0, "S": 1, "E": 1, "W": 0},
            (6, 0): {"N": 0, "S": 0, "E": 0, "W": 1}, (6, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (6, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (6, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (6, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (6, 5): {"N": 1, "S": 0, "E": 0, "W": 1}, (6, 6): {"N": 1, "S": 0, "E": 1, "W": 0}, (6, 7): {"N": 1, "S": 0, "E": 1, "W": 1},
            (7, 0): {"N": 0, "S": 1, "E": 1, "W": 1}, (7, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (7, 2): {"N": 0, "S": 1, "E": 0, "W": 0}, (7, 3): {"N": 1, "S": 1, "E": 0, "W": 0}, (7, 4): {"N": 1, "S": 1, "E": 0, "W": 0}, (7, 5): {"N": 0, "S": 1, "E": 1, "W": 0}, (7, 6): {"N": 0, "S": 1, "E": 0, "W": 1}, (7, 7): {"N": 0, "S": 1, "E": 1, "W": 0},
        },
    },
    {
        "name": "Maze 10: Stress Test",
        "size": (8, 8),
        "start": (0, 0),
        "exit": (7, 7),
        "optimal": 26,
        "limit": 55,
        "ascii": (
            "+--+--+--+--+--+--+--+--+\n"
            "|S    |     |           |\n"
            "+--+  +  +  +  +--+  +  +\n"
            "|     |  |        |  |  |\n"
            "+  +--+  +--+--+--+  +  +\n"
            "|     |     |        |  |\n"
            "+--+  +  +--+  +--+--+  +\n"
            "|  |  |  |     |     |  |\n"
            "+  +  +--+  +--+  +--+  +\n"
            "|  |        |        |  |\n"
            "+  +--+--+--+--+--+  +  +\n"
            "|        |           |  |\n"
            "+  +--+  +  +  +  +--+  +\n"
            "|  |  |  |  |  |  |     |\n"
            "+  +  +  +  +  +--+  +--+\n"
            "|     |     |         E |\n"
            "+--+--+--+--+--+--+--+--+"
        ),
        "grid": {
            (0, 0): {"N": 1, "S": 1, "E": 0, "W": 1}, (0, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 2): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 3): {"N": 1, "S": 0, "E": 1, "W": 0}, (0, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (0, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (0, 6): {"N": 1, "S": 0, "E": 0, "W": 0}, (0, 7): {"N": 1, "S": 0, "E": 1, "W": 0},
            (1, 0): {"N": 1, "S": 0, "E": 0, "W": 1}, (1, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (1, 2): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 3): {"N": 0, "S": 1, "E": 0, "W": 1}, (1, 4): {"N": 0, "S": 1, "E": 0, "W": 0}, (1, 5): {"N": 1, "S": 1, "E": 1, "W": 0}, (1, 6): {"N": 0, "S": 0, "E": 1, "W": 1}, (1, 7): {"N": 0, "S": 0, "E": 1, "W": 1},
            (2, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (2, 1): {"N": 1, "S": 0, "E": 1, "W": 0}, (2, 2): {"N": 0, "S": 0, "E": 0, "W": 1}, (2, 3): {"N": 1, "S": 1, "E": 1, "W": 0}, (2, 4): {"N": 1, "S": 0, "E": 0, "W": 1}, (2, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (2, 6): {"N": 0, "S": 1, "E": 1, "W": 0}, (2, 7): {"N": 0, "S": 0, "E": 1, "W": 1},
            (3, 0): {"N": 1, "S": 0, "E": 1, "W": 1}, (3, 1): {"N": 0, "S": 0, "E": 1, "W": 1}, (3, 2): {"N": 0, "S": 1, "E": 1, "W": 1}, (3, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 4): {"N": 0, "S": 1, "E": 1, "W": 0}, (3, 5): {"N": 1, "S": 0, "E": 0, "W": 1}, (3, 6): {"N": 1, "S": 1, "E": 1, "W": 0}, (3, 7): {"N": 0, "S": 0, "E": 1, "W": 1},
            (4, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (4, 1): {"N": 0, "S": 1, "E": 0, "W": 1}, (4, 2): {"N": 1, "S": 1, "E": 0, "W": 0}, (4, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (4, 4): {"N": 1, "S": 1, "E": 0, "W": 1}, (4, 5): {"N": 0, "S": 1, "E": 0, "W": 0}, (4, 6): {"N": 1, "S": 0, "E": 1, "W": 0}, (4, 7): {"N": 0, "S": 0, "E": 1, "W": 1},
            (5, 0): {"N": 0, "S": 0, "E": 0, "W": 1}, (5, 1): {"N": 1, "S": 1, "E": 0, "W": 0}, (5, 2): {"N": 1, "S": 0, "E": 1, "W": 0}, (5, 3): {"N": 1, "S": 0, "E": 0, "W": 1}, (5, 4): {"N": 1, "S": 0, "E": 0, "W": 0}, (5, 5): {"N": 1, "S": 0, "E": 0, "W": 0}, (5, 6): {"N": 0, "S": 1, "E": 1, "W": 0}, (5, 7): {"N": 0, "S": 0, "E": 1, "W": 1},
            (6, 0): {"N": 0, "S": 0, "E": 1, "W": 1}, (6, 1): {"N": 1, "S": 0, "E": 1, "W": 1}, (6, 2): {"N": 0, "S": 0, "E": 1, "W": 1}, (6, 3): {"N": 0, "S": 0, "E": 1, "W": 1}, (6, 4): {"N": 0, "S": 0, "E": 1, "W": 1}, (6, 5): {"N": 0, "S": 1, "E": 1, "W": 1}, (6, 6): {"N": 1, "S": 0, "E": 0, "W": 1}, (6, 7): {"N": 0, "S": 1, "E": 1, "W": 0},
            (7, 0): {"N": 0, "S": 1, "E": 0, "W": 1}, (7, 1): {"N": 0, "S": 1, "E": 1, "W": 0}, (7, 2): {"N": 0, "S": 1, "E": 0, "W": 1}, (7, 3): {"N": 0, "S": 1, "E": 1, "W": 0}, (7, 4): {"N": 0, "S": 1, "E": 0, "W": 1}, (7, 5): {"N": 1, "S": 1, "E": 0, "W": 0}, (7, 6): {"N": 0, "S": 1, "E": 0, "W": 0}, (7, 7): {"N": 1, "S": 1, "E": 1, "W": 0},
        },
    },
]


# --------------------------------------------------------------------------------
# 2. HELPER: Build Structured Wall List String for Player 2
# --------------------------------------------------------------------------------
def build_wall_list(maze_data):
    """Generate a structured, parseable wall description for every cell."""
    rows, cols = maze_data["size"]
    lines = []
    for r in range(rows):
        for c in range(cols):
            walls = maze_data["grid"][(r, c)]
            wall_dirs = [d for d in ("N", "S", "E", "W") if walls[d] == 1]
            open_dirs = [d for d in ("N", "S", "E", "W") if walls[d] == 0]
            label = ""
            if (r, c) == maze_data["exit"]:
                label = " [EXIT]"
            lines.append(
                f"  Cell ({r},{c}): walls=[{','.join(wall_dirs)}] open=[{','.join(open_dirs)}]{label}"
            )
    return "\n".join(lines)


# --------------------------------------------------------------------------------
# 3. HELPER: Truncate Conversation History (~3000 tokens ≈ ~12000 chars)
# --------------------------------------------------------------------------------
MAX_HISTORY_CHARS = 12000


def truncate_history(history_lines):
    """Keep the most recent turns if history exceeds the character budget."""
    full = "\n".join(history_lines)
    if len(full) <= MAX_HISTORY_CHARS:
        return full
    # Drop oldest turns until under budget
    while len("\n".join(history_lines)) > MAX_HISTORY_CHARS and len(history_lines) > 1:
        history_lines.pop(0)
    return "\n".join(history_lines)


# --------------------------------------------------------------------------------
# 4. SYSTEM PROMPTS (Matching Specification)
# --------------------------------------------------------------------------------
P1_SYSTEM_PROMPT = (
    "You are Player 1 (Navigator) in a cooperative maze game.\n"
    "You are inside a maze and can ONLY see the walls immediately around you "
    "(north, south, east, west — each is either 'wall' or 'open').\n"
    "Your partner (Player 2, the Cartographer) has the full map but does NOT know "
    "where you are. Only you can see your surroundings.\n\n"
    "Your goal: reach the exit in as few moves as possible.\n\n"
    "Each turn:\n"
    "1. You receive your wall perception and any message from the Cartographer.\n"
    "2. Communicate clearly what you see and any useful observations.\n"
    "3. Choose an action.\n\n"
    "You MUST respond in EXACTLY this format (two lines):\n"
    "MESSAGE: <your message to the Cartographer>\n"
    "ACTION: <move_north|move_south|move_east|move_west|stay>\n\n"
    "Tips:\n"
    "- Describe your walls precisely so the Cartographer can locate you.\n"
    "- Follow the Cartographer's guidance — they can see the full map.\n"
    "- Use 'stay' if you want to share information without moving.\n"
)

P2_SYSTEM_PROMPT_TEMPLATE = (
    "You are Player 2 (Cartographer) in a cooperative maze game.\n"
    "You have the COMPLETE map of the maze. Your partner (Player 1, the Navigator) "
    "is somewhere in this maze but you do NOT know where.\n"
    "The Navigator can only see the walls immediately around them.\n\n"
    "Your goal: figure out where the Navigator is using their wall descriptions, "
    "then guide them to the exit in as few moves as possible.\n\n"
    "=== MAZE MAP (Visual) ===\n{ascii}\n\n"
    "=== MAZE MAP (Structured) ===\n"
    "Grid: {rows}x{cols}, coordinates (row, col), 0-indexed from top-left.\n"
    "Exit: ({exit_r},{exit_c})\n"
    "{wall_list}\n\n"
    "Strategy:\n"
    "- Compare the Navigator's wall description against ALL cells to find matches.\n"
    "- If multiple cells match, ask them to move in a specific direction and report "
    "new walls to narrow down their position.\n"
    "- Once localized, compute the shortest path to the exit and give step-by-step "
    "directions.\n\n"
    "You MUST respond in EXACTLY this format:\n"
    "MESSAGE: <your guidance to the Navigator>\n"
)


def build_p2_system_prompt(maze_data):
    """Build the full Cartographer system prompt with maze data embedded."""
    exit_r, exit_c = maze_data["exit"]
    return P2_SYSTEM_PROMPT_TEMPLATE.format(
        ascii=maze_data["ascii"],
        rows=maze_data["size"][0],
        cols=maze_data["size"][1],
        exit_r=exit_r,
        exit_c=exit_c,
        wall_list=build_wall_list(maze_data),
    )


# --------------------------------------------------------------------------------
# 5. RESPONSE PARSERS
# --------------------------------------------------------------------------------
def parse_p1_response(raw):
    """Extract MESSAGE and ACTION from Player 1's response.
    On failure: returns empty message and 'stay'."""
    message = ""
    action = "stay"
    try:
        if "MESSAGE:" in raw and "ACTION:" in raw:
            message = raw.split("MESSAGE:")[1].split("ACTION:")[0].strip()
            action_raw = raw.split("ACTION:")[1].strip().split()[0].lower()
            if action_raw in ("move_north", "move_south", "move_east", "move_west", "stay"):
                action = action_raw
    except Exception:
        pass
    return message, action


def parse_p2_response(raw):
    """Extract MESSAGE from Player 2's response.
    On failure: returns empty message."""
    try:
        if "MESSAGE:" in raw:
            return raw.split("MESSAGE:")[1].strip()
    except Exception:
        pass
    return ""


# --------------------------------------------------------------------------------
# 6. ORCHESTRATOR: Execute Action
# --------------------------------------------------------------------------------
DIRECTION_MAP = {
    "move_north": ("N", -1, 0),
    "move_south": ("S", 1, 0),
    "move_east": ("E", 0, 1),
    "move_west": ("W", 0, -1),
}


def execute_action(maze_data, curr_pos, action):
    """Execute an action, return (new_pos, result_description).
    Handles wall collisions and stay."""
    if action == "stay":
        return curr_pos, "You stayed in place."

    if action not in DIRECTION_MAP:
        return curr_pos, "Invalid action. You stayed in place."

    wall_key, dr, dc = DIRECTION_MAP[action]
    r, c = curr_pos

    if maze_data["grid"][(r, c)][wall_key] == 1:
        return curr_pos, "You walked into a wall and stayed in place."

    new_pos = (r + dr, c + dc)
    direction_name = action.replace("move_", "")
    return new_pos, f"You moved {direction_name} successfully."


# --------------------------------------------------------------------------------
# 7. PERCEPTION BUILDER
# --------------------------------------------------------------------------------
def build_perception(maze_data, pos):
    """Build the wall perception string for Player 1."""
    walls = maze_data["grid"][pos]
    parts = []
    for d, label in [("N", "North"), ("S", "South"), ("E", "East"), ("W", "West")]:
        parts.append(f"  {label}: {'wall' if walls[d] == 1 else 'open'}")
    return "\n".join(parts)


# --------------------------------------------------------------------------------
# 8. UI RENDERER
# --------------------------------------------------------------------------------
def render_ui(maze_data, p1_pos, turn, p1_msg, p2_msg, move_result, status="playing"):
    rows, cols = maze_data["size"]
    cell_size = 48
    gap = 2

    # Status colors
    status_bg = "#0d1117"
    if status == "solved":
        status_bg = "#0a1f0a"
    elif status == "failed":
        status_bg = "#1f0a0a"

    # Build maze cells
    cells_html = ""
    for r in range(rows):
        for c in range(cols):
            walls = maze_data["grid"][(r, c)]
            border_n = "3px solid #58a6ff" if walls["N"] else "1px solid #21262d"
            border_s = "3px solid #58a6ff" if walls["S"] else "1px solid #21262d"
            border_e = "3px solid #58a6ff" if walls["E"] else "1px solid #21262d"
            border_w = "3px solid #58a6ff" if walls["W"] else "1px solid #21262d"

            content = ""
            bg = "#161b22"
            if (r, c) == p1_pos and (r, c) == maze_data["exit"]:
                content = "🎉"
                bg = "#238636"
            elif (r, c) == p1_pos:
                content = '<div style="width:22px;height:22px;background:#58a6ff;border-radius:50%;box-shadow:0 0 8px #58a6ff;"></div>'
                bg = "#1a2332"
            elif (r, c) == maze_data["exit"]:
                content = "🏁"
            elif (r, c) == maze_data["start"]:
                content = '<span style="color:#484f58;font-size:10px;">S</span>'

            cells_html += (
                f'<div style="width:{cell_size}px;height:{cell_size}px;background:{bg};'
                f"border-top:{border_n};border-bottom:{border_s};"
                f"border-right:{border_e};border-left:{border_w};"
                f'display:flex;align-items:center;justify-content:center;font-size:18px;">'
                f"{content}</div>"
            )

    # Score display for current state
    score_text = ""
    if status == "solved":
        score = maze_data["optimal"] / turn
        score_text = f'<div style="color:#3fb950;font-size:14px;margin-top:6px;">✅ Solved in {turn} turns — Score: {score:.3f}</div>'
    elif status == "failed":
        score_text = f'<div style="color:#f85149;font-size:14px;margin-top:6px;">❌ Failed — Score: 0.000</div>'

    # Move result indicator
    result_color = "#8b949e"
    if "wall" in move_result.lower():
        result_color = "#f85149"
    elif "successfully" in move_result.lower():
        result_color = "#3fb950"
    elif "stayed" in move_result.lower():
        result_color = "#d29922"

    empty_placeholder = '<i style="color:#484f58;">—</i>'
    p1_display = p1_msg if p1_msg else empty_placeholder
    p2_display = p2_msg if p2_msg else empty_placeholder

    html = f"""
    <div style="background:{status_bg};padding:16px;border-radius:10px;border:1px solid #30363d;
                font-family:'Segoe UI',system-ui,sans-serif;color:#c9d1d9;margin:6px 0;max-width:720px;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
            <span style="font-weight:600;font-size:15px;color:#f0f6fc;">{maze_data['name']}</span>
            <span style="font-size:13px;color:#8b949e;">Turn {turn}/{maze_data['limit']} · Optimal: {maze_data['optimal']}</span>
        </div>
        <div style="display:flex;gap:16px;">
            <div>
                <div style="display:grid;grid-template-columns:repeat({cols},{cell_size}px);gap:{gap}px;">
                    {cells_html}
                </div>
                {score_text}
            </div>
            <div style="flex:1;font-size:12px;line-height:1.5;min-width:200px;">
                <div style="color:{result_color};font-size:11px;padding:4px 8px;background:#21262d;
                            border-radius:4px;margin-bottom:8px;">⚡ {move_result}</div>
                <div style="margin-bottom:8px;">
                    <span style="color:#58a6ff;font-weight:600;font-size:11px;">🧭 NAVIGATOR</span><br/>
                    <span style="color:#c9d1d9;">{p1_display}</span>
                </div>
                <div>
                    <span style="color:#d29922;font-weight:600;font-size:11px;">🗺️ CARTOGRAPHER</span><br/>
                    <span style="color:#c9d1d9;">{p2_display}</span>
                </div>
            </div>
        </div>
    </div>"""
    return html


# --------------------------------------------------------------------------------
# 9. THE BENCHMARK TASK
# --------------------------------------------------------------------------------
@kbench.task(name="asymmetric_maze_solvers")
def maze_coop_benchmark(llm) -> float:
    maze_scores = []

    for maze in MAZE_SUITE:
        curr_pos = maze["start"]
        p1_history = []  # Conversation history from Player 1's perspective
        p2_history = []  # Conversation history from Player 2's perspective
        p2_system = build_p2_system_prompt(maze)
        p2_last_msg = ""
        last_move_result = "Game start. You have not moved yet."
        solved = False

        for turn in range(1, maze["limit"] + 1):
            # ── STEP 1: Build Player 1 prompt with perception + history ──
            perception = build_perception(maze, curr_pos)

            p1_turn_context = (
                f"--- Turn {turn} of {maze['limit']} ---\n"
                f"Your surroundings:\n{perception}\n"
                f"Last move result: {last_move_result}\n"
            )
            if p2_last_msg:
                p1_turn_context += f"Message from Cartographer: {p2_last_msg}\n"
            else:
                p1_turn_context += "No message from Cartographer yet.\n"

            p1_history.append(p1_turn_context)

            p1_full_prompt = (
                P1_SYSTEM_PROMPT
                + "\n=== CONVERSATION HISTORY ===\n"
                + truncate_history(list(p1_history))
                + "\n\nNow respond with your MESSAGE and ACTION for this turn."
            )

            # ── STEP 2: Player 1 responds (isolated chat) ──
            with kbench.chats.new(f"p1_{maze['name']}_{turn}"):
                p1_raw = llm.prompt(p1_full_prompt)
            p1_msg, action = parse_p1_response(p1_raw)

            # ── STEP 3: Orchestrator executes action ──
            curr_pos, last_move_result = execute_action(maze, curr_pos, action)

            # Record Player 1's response in history
            p1_history.append(f"Your message: {p1_msg}\nYour action: {action}\nResult: {last_move_result}")

            # ── Check for exit ──
            if curr_pos == maze["exit"]:
                solved = True
                score = maze["optimal"] / turn
                maze_scores.append(score)
                display(HTML(render_ui(maze, curr_pos, turn, p1_msg, p2_last_msg, last_move_result, "solved")))
                break

            # ── STEP 4: Build Player 2 prompt with history ──
            p2_turn_context = (
                f"--- Turn {turn} of {maze['limit']} ---\n"
                f"Navigator's message: {p1_msg if p1_msg else '(no message — parsing failed)'}\n"
                f"Navigator's action result: {last_move_result}\n"
            )
            p2_history.append(p2_turn_context)

            p2_full_prompt = (
                p2_system
                + "\n=== CONVERSATION HISTORY ===\n"
                + truncate_history(list(p2_history))
                + "\n\nNow respond with your MESSAGE for this turn."
            )

            # ── STEP 5: Player 2 responds (isolated chat) ──
            with kbench.chats.new(f"p2_{maze['name']}_{turn}"):
                p2_raw = llm.prompt(p2_full_prompt)
            p2_last_msg = parse_p2_response(p2_raw)

            # Record Player 2's response in history
            p2_history.append(f"Your message: {p2_last_msg}")

            # ── Render UI for observer ──
            display(HTML(render_ui(maze, curr_pos, turn, p1_msg, p2_last_msg, last_move_result)))

        # Maze not solved within limit
        if not solved:
            maze_scores.append(0.0)
            display(HTML(render_ui(maze, curr_pos, turn, p1_msg, p2_last_msg, last_move_result, "failed")))

    # ── Final Score ──
    final_score = sum(maze_scores) / len(MAZE_SUITE)

    display(HTML(
        f'<div style="background:#0d1117;padding:20px;border-radius:10px;border:1px solid #58a6ff;'
        f'font-family:monospace;color:#f0f6fc;text-align:center;margin:12px 0;max-width:720px;">'
        f'<div style="font-size:13px;color:#8b949e;margin-bottom:4px;">ASYMMETRIC MAZE COOPERATION</div>'
        f'<div style="font-size:32px;font-weight:700;color:#58a6ff;">{final_score:.4f}</div>'
        f'<div style="font-size:13px;color:#8b949e;margin-top:4px;">Final Score (0.0 – 1.0)</div>'
        f'<div style="font-size:11px;color:#484f58;margin-top:8px;">'
        f'{sum(1 for s in maze_scores if s > 0)}/{len(MAZE_SUITE)} mazes solved · '
        f'Scores: [{", ".join(f"{s:.3f}" for s in maze_scores)}]</div>'
        f'</div>'
    ))

    return final_score


# --------------------------------------------------------------------------------
# 10. EXECUTION
# --------------------------------------------------------------------------------
maze_coop_benchmark.run(kbench.llm)

In [ ]:
%choose maze_coop_benchmark